# 사투리 → 표준어 변환 모델 (KoBART 파인튜닝)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ysb2152/translate/blob/main/notebooks/kobart_finetune.ipynb)

2단계 변환기(사투리 텍스트 → 표준어 텍스트)를 KoBART seq2seq 로 파인튜닝한다. 백엔드의 규칙 스텁(`DialectConverter`)을 이 학습 모델로 교체하는 것이 목표.

**입력 데이터**: `data/build_mt_dataset.py`로 만든 균형 세트 `mt_balanced/{train,val}.jsonl`(`{dialect, standard}`). 텍스트라 작아서 지오블록·용량 문제 없이 Google Drive에 올려 쓰면 된다.

**준비**: 런타임 → GPU. 로컬에서 만든 `mt_balanced/` 폴더를 Google Drive `MyDrive/saturi/mt_balanced/` 에 업로드.

In [ ]:
!pip -q install -U "transformers>=4.44" datasets jiwer accelerate sentencepiece
import transformers, torch
print('transformers', transformers.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/saturi/mt_balanced'   # train.jsonl, val.jsonl 위치
OUT_DIR  = '/content/drive/MyDrive/saturi/kobart-dialect' # 학습 모델 저장 위치
MODEL_NAME = 'gogamza/kobart-base-v2'
import os; assert os.path.exists(f'{DATA_DIR}/train.jsonl'), f'{DATA_DIR}/train.jsonl 없음 — Drive에 업로드했는지 확인'
print('data', DATA_DIR, '| out', OUT_DIR)

In [ ]:
# 데이터 로드 + 토크나이즈
from datasets import load_dataset
from transformers import AutoTokenizer

ds = load_dataset('json', data_files={'train': f'{DATA_DIR}/train.jsonl',
                                      'validation': f'{DATA_DIR}/val.jsonl'})
print(ds)
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
MAXLEN = 128

def prep(batch):
    x = tok(batch['dialect'], max_length=MAXLEN, truncation=True)
    y = tok(text_target=batch['standard'], max_length=MAXLEN, truncation=True)
    x['labels'] = y['input_ids']
    return x

tokenized = ds.map(prep, batched=True, remove_columns=ds['train'].column_names)

In [ ]:
# 학습
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tok, model=model)

args = Seq2SeqTrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    num_train_epochs=2,          # 빠른 첫 실험은 1, 데이터 줄이려면 build 때 --max-train
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=200,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
)
trainer = Seq2SeqTrainer(model=model, args=args,
                         train_dataset=tokenized['train'],
                         eval_dataset=tokenized['validation'],
                         data_collator=collator)
trainer.train()

In [ ]:
# 평가: 학습 모델 vs '그대로 베끼기(copy)' 기준선 — CER과 정확일치
import jiwer, torch
model.eval()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; model.to(dev)

N = 500
sub = ds['validation'].select(range(min(N, len(ds['validation']))))
preds = []
for i in range(0, len(sub), 32):
    chunk = sub[i:i+32]['dialect']
    enc = tok(chunk, return_tensors='pt', padding=True, truncation=True, max_length=MAXLEN).to(dev)
    with torch.no_grad():
        gen = model.generate(**enc, max_length=MAXLEN, num_beams=4)
    preds += tok.batch_decode(gen, skip_special_tokens=True)

refs = sub['standard']; inps = sub['dialect']
def em(a, b):
    return sum(x.strip()==y.strip() for x,y in zip(a,b))/len(a)
print(f"CER  copy(사투리→표준)  : {jiwer.cer(refs, inps):.4f}")
print(f"CER  KoBART            : {jiwer.cer(refs, preds):.4f}")
print(f"정확일치 copy          : {em(inps, refs):.3f}")
print(f"정확일치 KoBART        : {em(preds, refs):.3f}")
print('\n--- 예시(방언 → 예측 / 정답) ---')
for d, p, r in list(zip(inps, preds, refs))[:8]:
    if d != r:
        print('방언:', d); print('예측:', p); print('정답:', r); print()

In [ ]:
# 저장(백엔드에서 로드할 폴더). tokenizer + model 함께 저장.
trainer.save_model(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print('저장 완료:', OUT_DIR)
!ls -la {OUT_DIR}

## 백엔드 연결

저장된 `kobart-dialect/` 폴더(모델+토크나이저)를 백엔드로 가져와 규칙 스텁을 교체한다.

1. Drive의 `kobart-dialect/` 를 내려받아 백엔드가 접근 가능한 경로에 둔다.
2. 백엔드 실행 시 환경변수로 경로 지정:
   ```bash
   set CONVERTER_MODEL_DIR=C:\path\to\kobart-dialect   # PowerShell: $env:CONVERTER_MODEL_DIR=...
   uvicorn app.main:app --host 0.0.0.0 --port 8000
   ```
3. `backend/requirements.txt` 의 torch/transformers 주석을 해제해 설치.

`DialectConverter` 는 `CONVERTER_MODEL_DIR` 가 있으면 KoBART로 추론하고, 없으면 규칙 스텁으로 폴백한다(B-9 구현).